# IV/TC Python Workflow (ASCAT + SMOS-IC + Model)

This notebook is a Python replacement for the MATLAB pipeline:
- Step 2: daily paired files (`sm_mod`, `sm_obs`, `idx_EASEv2_lonxlat`)
- Step 3: pentad climatology (`*_clim_pentad_*_w31.mat`)
- Step 4: IVD/IVS stats (`*_IVD_IVS_stats_*.mat`)
- TC: ASCAT + SMOS-IC + model (`ASCL4_SMOSIC_*_TC_stats_*.mat`)
- Step 5: R-diff export (`Rdiff_*_ASCL4.mat`, `Rdiff_*_SMOSIC.mat`)

Notes:
- MATLAB `griddata(...,'natural')` is approximated here with SciPy `linear` + optional nearest fill.
- The default date loop is **end-exclusive** (`start <= day < end`) to match most MATLAB loops in this repo.


In [ ]:
from __future__ import annotations

import sys
import os
from pathlib import Path
from datetime import date, timedelta
from dataclasses import dataclass

# If your platform needs OpenMP guards, uncomment before importing numpy/scipy:
# os.environ.setdefault('OMP_NUM_THREADS', '1')
# os.environ.setdefault('MKL_NUM_THREADS', '1')
# os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
import scipy.io as sio
from scipy.interpolate import griddata
from netCDF4 import Dataset
import h5py

# Repo-local helpers
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'common').exists():
    REPO_ROOT = Path('/discover/nobackup/projects/land_da/geosldas-analysis')

sys.path.insert(0, str(REPO_ROOT / 'common/python/io'))
sys.path.insert(0, str(REPO_ROOT / 'projects/matlab2python/shared/python'))

from read_GEOSldas import read_tilecoord  # type: ignore
from EASEv2 import EASEv2_ind2latlon      # type: ignore

print(f'REPO_ROOT={REPO_ROOT}')


In [ ]:
@dataclass
class WorkflowConfig:
    # Date range: end date is treated as exclusive in loops
    start_date: date = date(2018, 8, 1)
    end_date: date = date(2024, 6, 30)

    # Domain / model output
    domain: str = 'SMAP_EASEv2_M36_GLOBAL'
    out_collection: str = '.tavg24_1d_lnd_Nt.'

    # Run roots (edit as needed)
    run_roots: dict[str, Path] = None

    # Inputs
    ascat_root: Path = Path('/discover/nobackup/qliu/merra_land/DATA/ASCAT_HSAF')
    smosic_preprocessed_root: Path = Path('/discover/nobackup/projects/land_da/SMOS_IC/preprocessed_m36_daily')

    # Outputs
    ivs_output_root: Path = Path('/discover/nobackup/projects/land_da/Evaluation/IVs/output_python')

    # Step options
    ascat_interp_method: str = 'linear'   # 'linear' or 'nearest'
    ascat_fill_linear_nans_with_nearest: bool = True
    nlag_days: int = 2
    nmin_ivs: int = 100
    nmin_tc: int = 20

    # TC season flags
    do_year_tc: bool = True
    do_summer_tc: bool = False


def default_cfg() -> WorkflowConfig:
    roots = {
        'OLv8_M36_cd': Path('/discover/nobackup/projects/land_da/CYGNSS_Experiments/OLv8_M36_cd'),
        'DAv8_M36_cd': Path('/discover/nobackup/projects/land_da/CYGNSS_Experiments/DAv8_M36_cd'),
    }
    cfg = WorkflowConfig(run_roots=roots)
    cfg.ivs_output_root.mkdir(parents=True, exist_ok=True)
    return cfg


cfg = default_cfg()
print(cfg)


In [ ]:
# ---------- Date/grid/path helpers ----------

def daterange(start: date, end_exclusive: date):
    d = start
    while d < end_exclusive:
        yield d
        d += timedelta(days=1)


def date_str(d: date) -> str:
    return f'{d.year:04d}{d.month:02d}{d.day:02d}'


def matlab_time_tag(start_d: date, end_d: date) -> str:
    # mirrors MATLAB scripts that use end month - 1 for tag
    if end_d.month == 1:
        return f'{start_d.year}{start_d.month:02d}_{end_d.year-1}12'
    return f'{start_d.year}{start_d.month:02d}_{end_d.year}{end_d.month-1:02d}'


def dofyr_nonleap(d: date) -> int:
    # MATLAB logic: map leap day to 2/28 in a non-leap reference year
    if d.month == 2 and d.day == 29:
        d2 = date(2017, 2, 28)
    else:
        d2 = date(2017, d.month, d.day)
    return int(d2.strftime('%j'))


def pentad_1based(d: date) -> int:
    return (dofyr_nonleap(d) - 1) // 5 + 1  # 1..73


PENTAD_CENTERS_DOY = np.arange(3, 366, 5, dtype=np.int32)  # 73 centers


def circular_day_distance(a: int, b: int) -> int:
    x = abs(a - b)
    return min(x, 365 - x)


def build_doy_to_pentad_window_map(window_days: int = 15):
    # for each doy(1..365), list pentad-center indices (0..72) within +/-window_days
    out = {}
    for doy in range(1, 366):
        keep = [i for i, c in enumerate(PENTAD_CENTERS_DOY) if circular_day_distance(doy, int(c)) <= window_days]
        out[doy] = np.array(keep, dtype=np.int32)
    return out


DOY_TO_PENTADS = build_doy_to_pentad_window_map(15)


def ease2_m36_lon_lat():
    # Build lon/lat as MATLAB style: shape (Nlon, Nlat) = (964, 406)
    nlon, nlat = 964, 406
    row2d = np.repeat(np.arange(nlat, dtype=np.float64)[None, :], nlon, axis=0)
    col2d = np.repeat(np.arange(nlon, dtype=np.float64)[:, None], nlat, axis=1)
    lat, lon = EASEv2_ind2latlon(row2d, col2d, 'M36')
    return lon.astype(np.float64), lat.astype(np.float64)


def tilecoord_path_for_run(run_root: Path, run_name: str, domain: str) -> Path:
    cands = [
        run_root / 'output' / domain / 'rc_out' / f'{run_name}.ldas_tilecoord.bin',
        run_root / run_name / 'output' / domain / 'rc_out' / f'{run_name}.ldas_tilecoord.bin',
        run_root / f'{run_name}.ldas_tilecoord.bin',
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find tilecoord. Checked: ' + ', '.join(str(p) for p in cands))


def read_model_tilecoord(run_root: Path, run_name: str, domain: str):
    p = tilecoord_path_for_run(run_root, run_name, domain)
    tc = read_tilecoord(str(p))
    # Keep as zero-based integers for numpy indexing
    i = np.asarray(tc['i_indg'], dtype=np.int64)
    j = np.asarray(tc['j_indg'], dtype=np.int64)
    n_tile = int(tc['N_tile'])
    return {'path': p, 'i_indg': i, 'j_indg': j, 'N_tile': n_tile}


def load_mat_fields(path: Path, fields: tuple[str, ...]):
    """Robust mat reader for v7 (scipy) and v7.3 (h5py)."""
    try:
        d = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
        out = {}
        for f in fields:
            if f not in d:
                raise KeyError(f'Missing field {f} in {path}')
            out[f] = np.asarray(d[f])
        return out
    except NotImplementedError:
        out = {}
        with h5py.File(path, 'r') as h5:
            for f in fields:
                if f not in h5:
                    raise KeyError(f'Missing field {f} in {path}')
                out[f] = np.asarray(h5[f]).squeeze()
        return out


def save_daily_pair_mat(path: Path, sm_mod_2d: np.ndarray, sm_obs_2d: np.ndarray):
    # Apply mutual finite mask
    sm_obs = sm_obs_2d.copy()
    sm_mod = sm_mod_2d.copy()
    sm_obs[~np.isfinite(sm_mod)] = np.nan
    sm_mod[~np.isfinite(sm_obs)] = np.nan

    flat_obs = sm_obs.ravel(order='F')
    flat_mod = sm_mod.ravel(order='F')

    idx0 = np.flatnonzero(np.isfinite(flat_obs))
    idx1 = (idx0 + 1).astype(np.int32)  # MATLAB 1-based linear indexing

    if idx0.size == 0:
        sm_obs_v = np.empty((0, 1), dtype=np.float64)
        sm_mod_v = np.empty((0, 1), dtype=np.float64)
        idx1_v = np.empty((0, 1), dtype=np.int32)
    else:
        sm_obs_v = flat_obs[idx0].astype(np.float64)[:, None]
        sm_mod_v = flat_mod[idx0].astype(np.float64)[:, None]
        idx1_v = idx1[:, None]

    sio.savemat(path, {
        'sm_mod': sm_mod_v,
        'sm_obs': sm_obs_v,
        'idx_EASEv2_lonxlat': idx1_v,
    }, do_compression=True)


def load_daily_pair_mat(path: Path):
    d = load_mat_fields(path, ('sm_mod', 'sm_obs', 'idx_EASEv2_lonxlat'))
    sm_mod = np.asarray(d['sm_mod'], dtype=np.float64).reshape(-1)
    sm_obs = np.asarray(d['sm_obs'], dtype=np.float64).reshape(-1)
    idx1 = np.asarray(d['idx_EASEv2_lonxlat']).reshape(-1).astype(np.int64)
    idx0 = idx1 - 1
    return idx0, sm_mod, sm_obs


lon_L4, lat_L4 = ease2_m36_lon_lat()
Nlon, Nlat = lon_L4.shape
Ncell = Nlon * Nlat
print(f'M36 grid: Nlon={Nlon}, Nlat={Nlat}, Ncell={Ncell}')


In [ ]:
# ---------- Step 2: build daily pair mats ----------

def model_daily_file_candidates(run_root: Path, run_name: str, domain: str, out_collection: str, d: date):
    y = f'{d.year:04d}'
    m = f'{d.month:02d}'
    day = f'{d.day:02d}'

    cands = []
    # aggregated daily first (MATLAB order)
    cands.append(run_root / 'output' / domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}_1200z.nc4')
    cands.append(run_root / 'output' / domain / 'cat' / 'ens_avg'  / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}_1200z.nc4')
    cands.append(run_root / 'output' / domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}.nc4')
    cands.append(run_root / 'output' / domain / 'cat' / 'ens_avg'  / f'Y{y}' / f'M{m}' / f'{run_name}{out_collection}{y}{m}{day}.nc4')
    return cands


def read_nc_var(path: Path, var: str):
    with Dataset(path, 'r') as ds:
        if var not in ds.variables:
            raise KeyError(f'Variable {var} not found in {path}')
        arr = np.array(ds.variables[var][:], dtype=np.float64)
    arr[arr < 0] = np.nan
    return arr


def orient_sfmc_for_tiles(arr: np.ndarray, n_tile: int):
    arr = np.asarray(arr, dtype=np.float64)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if arr.shape[0] != n_tile:
            raise RuntimeError(f'SFMC length {arr.shape[0]} != N_tile {n_tile}')
        return arr.reshape(n_tile, 1)

    if arr.ndim == 2:
        n1, n2 = arr.shape
        if n1 == n_tile:
            return arr
        if n2 == n_tile:
            return arr.T
        raise RuntimeError(f'Unexpected SFMC shape {arr.shape}, cannot map to N_tile={n_tile}')

    raise RuntimeError(f'Unexpected SFMC ndim={arr.ndim}, shape={arr.shape}')


def read_model_daily_mean_for_run(d: date, run_root: Path, run_name: str, cfg: WorkflowConfig, tc: dict):
    # 1) Try aggregated daily file + SFMC
    for p in model_daily_file_candidates(run_root, run_name, cfg.domain, cfg.out_collection, d):
        if p.exists():
            sfmc = read_nc_var(p, 'SFMC')
            sfmc2 = orient_sfmc_for_tiles(sfmc, tc['N_tile'])
            return np.nanmean(sfmc2, axis=1)

    # 2) Fallback: 8 hourly files, var sm_surface
    hr_list = [1, 4, 7, 10, 13, 16, 19, 22]
    rows = []
    y = f'{d.year:04d}'
    m = f'{d.month:02d}'
    day = f'{d.day:02d}'

    for hh in hr_list:
        hhmm = f'{hh:02d}30'
        name = f'{run_name}{cfg.out_collection}{y}{m}{day}_{hhmm}z.nc4'
        p1 = run_root / 'output' / cfg.domain / 'cat' / 'ens0000' / f'Y{y}' / f'M{m}' / name
        p2 = run_root / 'output' / cfg.domain / 'cat' / 'ens_avg' / f'Y{y}' / f'M{m}' / name
        p = p1 if p1.exists() else p2
        if not p.exists():
            raise FileNotFoundError(f'Missing hourly model file: {p1} OR {p2}')

        arr = read_nc_var(p, 'sm_surface').reshape(-1)
        if arr.shape[0] != tc['N_tile']:
            raise RuntimeError(f'sm_surface length {arr.shape[0]} != N_tile {tc["N_tile"]} in {p}')
        rows.append(arr)

    mat = np.stack(rows, axis=1)
    return np.nanmean(mat, axis=1)


def map_tile_vector_to_m36(sm_tile: np.ndarray, tc: dict, nlon: int = 964, nlat: int = 406):
    out = np.full((nlon, nlat), np.nan, dtype=np.float64)
    out[tc['i_indg'], tc['j_indg']] = sm_tile
    return out


def read_ascat_land_grid_info(ascat_root: Path):
    f = ascat_root / 'Auxiliary' / 'TUW_WARP5_grid_info_2_2.nc'
    with Dataset(f, 'r') as ds:
        land_flag = np.asarray(ds.variables['land_flag'][:]).reshape(-1)
        lon = np.asarray(ds.variables['lon'][:], dtype=np.float64).reshape(-1)
        lat = np.asarray(ds.variables['lat'][:], dtype=np.float64).reshape(-1)
    m = (land_flag == 1)
    return lon[m], lat[m]


def read_ascat_obs_on_m36(d: date, cfg: WorkflowConfig, lon_gpi_land: np.ndarray, lat_gpi_land: np.ndarray, lon_m36: np.ndarray, lat_m36: np.ndarray):
    f = (cfg.ascat_root / 'H119_H120_processed' / f'Y{d.year:04d}' / f'M{d.month:02d}' / f'ASCAT_HSAF_H119_SM_{date_str(d)}_AD.mat')

    sm_obs = np.full(lon_m36.shape, np.nan, dtype=np.float64)
    if not f.exists():
        return sm_obs

    dct = load_mat_fields(f, ('sm_tile', 'conf_flag_tile'))
    sm_asc = np.asarray(dct['sm_tile'], dtype=np.float64).reshape(-1)
    conf = np.asarray(dct['conf_flag_tile']).reshape(-1)

    if sm_asc.shape[0] != lon_gpi_land.shape[0]:
        raise RuntimeError(f'ASCAT sm_tile length {sm_asc.shape[0]} != land GPI length {lon_gpi_land.shape[0]} for {f}')

    sm_asc[sm_asc > 100] = np.nan
    sm_asc[conf >= 1] = np.nan

    v = np.isfinite(sm_asc)
    if np.count_nonzero(v) < 3:
        return sm_obs

    points = np.column_stack((lon_gpi_land[v], lat_gpi_land[v]))
    values = sm_asc[v]

    method = cfg.ascat_interp_method.lower()
    if method not in ('linear', 'nearest'):
        raise ValueError(f'Unsupported ascat_interp_method={cfg.ascat_interp_method}. Use linear/nearest.')

    sm_obs = griddata(points, values, (lon_m36, lat_m36), method=method)

    if method == 'linear' and cfg.ascat_fill_linear_nans_with_nearest:
        fill = griddata(points, values, (lon_m36, lat_m36), method='nearest')
        sm_obs = np.where(np.isfinite(sm_obs), sm_obs, fill)

    return sm_obs


def read_smosic_obs_on_m36(d: date, cfg: WorkflowConfig, nlon: int = 964, nlat: int = 406):
    f = cfg.smosic_preprocessed_root / f'smos_ic_sm_m36_{date_str(d)}.nc'
    out = np.full((nlon, nlat), np.nan, dtype=np.float64)

    if not f.exists():
        return out

    with Dataset(f, 'r') as ds:
        idx0 = np.asarray(ds.variables['idx_EASEv2_lonxlat'][:], dtype=np.int64).reshape(-1)
        vals = np.asarray(ds.variables['sm_obs'][:], dtype=np.float64).reshape(-1)

    good = (idx0 >= 0) & (idx0 < nlon * nlat) & np.isfinite(vals)
    if np.any(good):
        flat = out.ravel(order='F')
        flat[idx0[good]] = vals[good]
        out = flat.reshape((nlon, nlat), order='F')

    return out


def run_step2_for_sensor(sensor_prefix: str, run_name: str, run_root: Path, cfg: WorkflowConfig):
    if sensor_prefix not in ('ASCL4', 'SMOSIC'):
        raise ValueError('sensor_prefix must be ASCL4 or SMOSIC')

    tc = read_model_tilecoord(run_root, run_name, cfg.domain)
    nlon, nlat = lon_L4.shape

    lon_gpi_land = lat_gpi_land = None
    if sensor_prefix == 'ASCL4':
        lon_gpi_land, lat_gpi_land = read_ascat_land_grid_info(cfg.ascat_root)

    n_done = 0
    n_missing_obs = 0
    for d in daterange(cfg.start_date, cfg.end_date):
        sm_mod_tile = read_model_daily_mean_for_run(d, run_root, run_name, cfg, tc)
        sm_mod_2d = map_tile_vector_to_m36(sm_mod_tile, tc, nlon=nlon, nlat=nlat)

        if sensor_prefix == 'ASCL4':
            sm_obs_2d = read_ascat_obs_on_m36(d, cfg, lon_gpi_land, lat_gpi_land, lon_L4, lat_L4)
            out = cfg.ivs_output_root / f'ASCL4_H119_SMSF_L4_{run_name}_QC_1_{date_str(d)}.mat'
        else:
            sm_obs_2d = read_smosic_obs_on_m36(d, cfg, nlon=nlon, nlat=nlat)
            out = cfg.ivs_output_root / f'SMOSIC_SMSF_MOD_{run_name}_QC_1_{date_str(d)}.mat'

        if not np.isfinite(sm_obs_2d).any():
            n_missing_obs += 1

        save_daily_pair_mat(out, sm_mod_2d, sm_obs_2d)
        n_done += 1

        if n_done % 100 == 0:
            print(f'[{sensor_prefix} {run_name}] wrote {n_done} days...')

    print(f'[{sensor_prefix} {run_name}] complete: days={n_done}, days_with_all_nan_obs={n_missing_obs}')


In [ ]:
# ---------- Step 3: climatology ----------

def pair_file_for_prefix(prefix: str, run_name: str, d: date, out_root: Path):
    ds = date_str(d)
    if prefix == 'ASCL4':
        return out_root / f'ASCL4_H119_SMSF_L4_{run_name}_QC_1_{ds}.mat'
    if prefix == 'SMOSIC':
        return out_root / f'SMOSIC_SMSF_MOD_{run_name}_QC_1_{ds}.mat'
    raise ValueError(prefix)


def run_step3_climatology(prefix: str, run_name: str, cfg: WorkflowConfig, window_days: int = 15):
    # Memory-optimized: accumulate directly for 73 pentad centers (equivalent to 365->3:5:365 export)
    ncell = Ncell
    npen = len(PENTAD_CENTERS_DOY)  # 73

    mod_sum = np.zeros((ncell, npen), dtype=np.float32)
    obs_sum = np.zeros((ncell, npen), dtype=np.float32)
    n_sum = np.zeros((ncell, npen), dtype=np.int32)

    nday_min = 4 * (cfg.end_date.year - cfg.start_date.year)

    for k, d in enumerate(daterange(cfg.start_date, cfg.end_date), start=1):
        f = pair_file_for_prefix(prefix, run_name, d, cfg.ivs_output_root)
        if not f.exists():
            continue

        idx0, sm_mod, sm_obs = load_daily_pair_mat(f)
        if idx0.size == 0:
            continue

        doy = dofyr_nonleap(d)
        pidx = DOY_TO_PENTADS[doy]

        for p in pidx:
            mod_sum[idx0, p] += sm_mod.astype(np.float32)
            obs_sum[idx0, p] += sm_obs.astype(np.float32)
            n_sum[idx0, p] += 1

        if k % 200 == 0:
            print(f'[{prefix} {run_name}] step3 day {k}')

    count_f = n_sum.astype(np.float32)
    valid = count_f >= float(nday_min)

    mod_clim = np.full((ncell, npen), np.nan, dtype=np.float32)
    obs_clim = np.full((ncell, npen), np.nan, dtype=np.float32)

    mod_clim[valid] = mod_sum[valid] / count_f[valid]
    obs_clim[valid] = obs_sum[valid] / count_f[valid]

    tag = matlab_time_tag(cfg.start_date, cfg.end_date)
    fout = cfg.ivs_output_root / f'{prefix}_{run_name}_clim_pentad_{tag}_w31.mat'

    sio.savemat(fout, {
        'mod_sm_clim': mod_clim,
        'obs_sm_clim': obs_clim,
        'N_sm_clim': n_sum.astype(np.int32),
        'Nday_min': np.array([[nday_min]], dtype=np.int32),
    }, do_compression=True)

    print(f'Wrote: {fout}')
    return fout


In [ ]:
# ---------- Step 4: IVD/IVS ----------

def run_step4_ivd_ivs(prefix: str, run_name: str, cfg: WorkflowConfig):
    ncell = Ncell
    nlag = int(cfg.nlag_days)
    nmin = int(cfg.nmin_ivs)

    tag = matlab_time_tag(cfg.start_date, cfg.end_date)
    f_clim = cfg.ivs_output_root / f'{prefix}_{run_name}_clim_pentad_{tag}_w31.mat'
    c = load_mat_fields(f_clim, ('mod_sm_clim', 'obs_sm_clim'))
    mod_clim = np.asarray(c['mod_sm_clim'], dtype=np.float64)
    obs_clim = np.asarray(c['obs_sm_clim'], dtype=np.float64)

    mod_sm_sum = np.zeros(ncell, dtype=np.float64)
    mod_sm2_sum = np.zeros(ncell, dtype=np.float64)
    obs_sm_sum = np.zeros(ncell, dtype=np.float64)
    obs_sm2_sum = np.zeros(ncell, dtype=np.float64)

    modxobs_sm_sum = np.zeros(ncell, dtype=np.float64)
    modxlag1_sm_sum = np.zeros(ncell, dtype=np.float64)
    obsxlag1_sm_sum = np.zeros(ncell, dtype=np.float64)
    obslag1xmod_sm_sum = np.zeros(ncell, dtype=np.float64)
    modlag1xobs_sm_sum = np.zeros(ncell, dtype=np.float64)
    n_sm = np.zeros(ncell, dtype=np.int32)

    start2 = cfg.start_date + timedelta(days=nlag)

    for k, d in enumerate(daterange(start2, cfg.end_date), start=1):
        dpre = d - timedelta(days=nlag)
        f_now = pair_file_for_prefix(prefix, run_name, d, cfg.ivs_output_root)
        f_pre = pair_file_for_prefix(prefix, run_name, dpre, cfg.ivs_output_root)

        if not (f_now.exists() and f_pre.exists()):
            continue

        idx_now, mod_now, obs_now = load_daily_pair_mat(f_now)
        idx_pre, mod_pre, obs_pre = load_daily_pair_mat(f_pre)

        if idx_now.size == 0 or idx_pre.size == 0:
            continue

        idx, inow, ipre = np.intersect1d(idx_now, idx_pre, assume_unique=False, return_indices=True)
        if idx.size == 0:
            continue

        p_now = pentad_1based(d) - 1
        p_pre = pentad_1based(dpre) - 1

        sm_mod = mod_now[inow] - mod_clim[idx, p_now]
        sm_obs = obs_now[inow] - obs_clim[idx, p_now]
        sm_mod_pre = mod_pre[ipre] - mod_clim[idx, p_pre]
        sm_obs_pre = obs_pre[ipre] - obs_clim[idx, p_pre]

        iv = np.isfinite(sm_mod)  # MATLAB matches only on sm_mod finite
        if not np.any(iv):
            continue

        ii = idx[iv]
        m = sm_mod[iv]
        o = sm_obs[iv]
        mp = sm_mod_pre[iv]
        op = sm_obs_pre[iv]

        mod_sm_sum[ii] += m
        mod_sm2_sum[ii] += m * m
        obs_sm_sum[ii] += o
        obs_sm2_sum[ii] += o * o

        modxobs_sm_sum[ii] += m * o
        modxlag1_sm_sum[ii] += m * mp
        obsxlag1_sm_sum[ii] += o * op
        obslag1xmod_sm_sum[ii] += m * op
        modlag1xobs_sm_sum[ii] += mp * o

        n_sm[ii] += 1

        if k % 200 == 0:
            print(f'[{prefix} {run_name}] step4 day {k}')

    NN = n_sm.astype(np.float64)
    NN[NN < nmin] = np.nan

    mod_sm_mean = mod_sm_sum / NN
    obs_sm_mean = obs_sm_sum / NN

    mod_sm_mean[~np.isfinite(mod_sm_mean)] = np.nan
    obs_sm_mean[~np.isfinite(obs_sm_mean)] = np.nan

    C_mod_mod = mod_sm2_sum / NN - mod_sm_mean ** 2
    C_obs_obs = obs_sm2_sum / NN - obs_sm_mean ** 2
    C_mod_mod[C_mod_mod < 0.0] = np.nan
    C_obs_obs[C_obs_obs < 0.0] = np.nan

    C_mod_obs = modxobs_sm_sum / NN - mod_sm_mean * obs_sm_mean

    R_mod_obs = C_mod_obs / np.sqrt(C_mod_mod * C_obs_obs)
    R_mod_obs[~np.isfinite(R_mod_obs)] = -9999.0

    C_mod_obs[C_mod_obs < 0.0] = np.nan

    C_mod_modlag1 = modxlag1_sm_sum / NN - mod_sm_mean ** 2
    C_obs_obslag1 = obsxlag1_sm_sum / NN - obs_sm_mean ** 2
    C_mod_modlag1[C_mod_modlag1 < 0.0] = np.nan
    C_obs_obslag1[C_obs_obslag1 < 0.0] = np.nan

    C_mod_obslag1 = obslag1xmod_sm_sum / NN - mod_sm_mean * obs_sm_mean
    C_modlag1_obs = modlag1xobs_sm_sum / NN - mod_sm_mean * obs_sm_mean

    S_ivd = np.sqrt(C_mod_modlag1 / C_obs_obslag1)
    S_ivs_obs = C_mod_obslag1 / C_obs_obslag1
    S_ivs_mod = C_mod_modlag1 / C_modlag1_obs

    R2_ivd_mod = C_mod_obs * S_ivd / C_mod_mod
    R2_ivd_obs = C_mod_obs / C_obs_obs / S_ivd

    R2_ivs_mod = C_mod_obs * S_ivs_obs / C_mod_mod
    R2_ivs_obs = C_mod_obs / C_obs_obs / S_ivs_obs

    for arr in (R2_ivd_mod, R2_ivd_obs, R2_ivs_mod, R2_ivs_obs):
        arr[arr < 0.0] = np.nan
        arr[arr > 1.0] = 1.0

    fout = cfg.ivs_output_root / f'{prefix}_{run_name}_IVD_IVS_stats_lag{nlag}day_{tag}.mat'
    sio.savemat(fout, {
        'N_sm': n_sm.astype(np.int32),
        'Nmin': np.array([[nmin]], dtype=np.int32),
        'Nlag': np.array([[nlag]], dtype=np.int32),
        'R2_ivd_mod': R2_ivd_mod,
        'R2_ivd_obs': R2_ivd_obs,
        'R2_ivs_mod': R2_ivs_mod,
        'R2_ivs_obs': R2_ivs_obs,
        'R_mod_obs': R_mod_obs,
    }, do_compression=True)

    print(f'Wrote: {fout}')
    return fout


In [ ]:
# ---------- TC: ASCAT + SMOS-IC + model ----------

def run_tc_ascat_smosic(run_name: str, cfg: WorkflowConfig):
    ncell = Ncell
    nmin = int(cfg.nmin_tc)
    tag = matlab_time_tag(cfg.start_date, cfg.end_date)

    cA = load_mat_fields(cfg.ivs_output_root / f'ASCL4_{run_name}_clim_pentad_{tag}_w31.mat', ('mod_sm_clim', 'obs_sm_clim'))
    cS = load_mat_fields(cfg.ivs_output_root / f'SMOSIC_{run_name}_clim_pentad_{tag}_w31.mat', ('obs_sm_clim',))

    mod_clim = np.asarray(cA['mod_sm_clim'], dtype=np.float64)
    asc_clim = np.asarray(cA['obs_sm_clim'], dtype=np.float64) / 200.0
    smos_clim = np.asarray(cS['obs_sm_clim'], dtype=np.float64)

    SMOS_sm_sum = np.zeros(ncell, dtype=np.float64)
    SMOS_sm2_sum = np.zeros(ncell, dtype=np.float64)
    mod_sm_sum = np.zeros(ncell, dtype=np.float64)
    mod_sm2_sum = np.zeros(ncell, dtype=np.float64)
    ASC_sm_sum = np.zeros(ncell, dtype=np.float64)
    ASC_sm2_sum = np.zeros(ncell, dtype=np.float64)

    modxASC_sm_sum = np.zeros(ncell, dtype=np.float64)
    SMOSxASC_sm_sum = np.zeros(ncell, dtype=np.float64)
    SMOSxmod_sm_sum = np.zeros(ncell, dtype=np.float64)
    N_sm = np.zeros(ncell, dtype=np.int32)

    for k, d in enumerate(daterange(cfg.start_date, cfg.end_date), start=1):
        f_asc = pair_file_for_prefix('ASCL4', run_name, d, cfg.ivs_output_root)
        f_smos = pair_file_for_prefix('SMOSIC', run_name, d, cfg.ivs_output_root)

        if not (f_asc.exists() and f_smos.exists()):
            continue

        idx_asc, mod_asc, obs_asc = load_daily_pair_mat(f_asc)
        idx_smos, mod_smos, obs_smos = load_daily_pair_mat(f_smos)

        if idx_asc.size == 0 or idx_smos.size == 0:
            continue

        idx, ia, ib = np.intersect1d(idx_smos, idx_asc, assume_unique=False, return_indices=True)
        if idx.size == 0:
            continue

        sm_mod = mod_smos[ia]          # use SMOS-side model, as MATLAB script
        sm_SMOS = obs_smos[ia]
        sm_ASCAT = obs_asc[ib] / 200.0

        # optional seasonal subset (MATLAB logic)
        keep = np.ones(idx.shape[0], dtype=bool)
        if not cfg.do_year_tc:
            if cfg.do_summer_tc:
                keep = (d.month >= 5) & (d.month <= 9)
            else:
                keep = (d.month <= 4) | (d.month >= 10)
            if np.isscalar(keep):
                keep = np.full(idx.shape[0], bool(keep), dtype=bool)

        if not np.any(keep):
            continue

        idx = idx[keep]
        sm_mod = sm_mod[keep]
        sm_SMOS = sm_SMOS[keep]
        sm_ASCAT = sm_ASCAT[keep]

        p = pentad_1based(d) - 1

        sm_SMOS = sm_SMOS - smos_clim[idx, p]
        sm_mod = sm_mod - mod_clim[idx, p]
        sm_ASC = sm_ASCAT - asc_clim[idx, p]

        iv = np.isfinite(sm_mod) & np.isfinite(sm_SMOS) & np.isfinite(sm_ASC)
        if not np.any(iv):
            continue

        ii = idx[iv]
        m = sm_mod[iv]
        s = sm_SMOS[iv]
        a = sm_ASC[iv]

        mod_sm_sum[ii] += m
        mod_sm2_sum[ii] += m * m
        SMOS_sm_sum[ii] += s
        SMOS_sm2_sum[ii] += s * s
        ASC_sm_sum[ii] += a
        ASC_sm2_sum[ii] += a * a

        SMOSxASC_sm_sum[ii] += s * a
        SMOSxmod_sm_sum[ii] += s * m
        modxASC_sm_sum[ii] += m * a
        N_sm[ii] += 1

        if k % 200 == 0:
            print(f'[TC {run_name}] day {k}')

    NN = N_sm.astype(np.float64)
    NN[NN < nmin] = np.nan

    SMOS_sm_mean = SMOS_sm_sum / NN
    mod_sm_mean = mod_sm_sum / NN
    ASC_sm_mean = ASC_sm_sum / NN

    C_SMOS_SMOS = SMOS_sm2_sum / NN - SMOS_sm_mean ** 2
    C_mod_mod = mod_sm2_sum / NN - mod_sm_mean ** 2
    C_ASC_ASC = ASC_sm2_sum / NN - ASC_sm_mean ** 2

    C_SMOS_SMOS[C_SMOS_SMOS < 0.0] = np.nan
    C_mod_mod[C_mod_mod < 0.0] = np.nan
    C_ASC_ASC[C_ASC_ASC < 0.0] = np.nan

    C_mod_ASC = modxASC_sm_sum / NN - mod_sm_mean * ASC_sm_mean
    C_SMOS_mod = SMOSxmod_sm_sum / NN - SMOS_sm_mean * mod_sm_mean
    C_SMOS_ASC = SMOSxASC_sm_sum / NN - SMOS_sm_mean * ASC_sm_mean

    R_mod_SMOS = C_SMOS_mod / np.sqrt(C_mod_mod * C_SMOS_SMOS)
    R_mod_ASC = C_mod_ASC / np.sqrt(C_mod_mod * C_ASC_ASC)
    R_ASC_SMOS = C_SMOS_ASC / np.sqrt(C_SMOS_SMOS * C_ASC_ASC)

    R2_TC_SMOS = C_SMOS_mod * C_SMOS_ASC / C_mod_ASC / C_SMOS_SMOS
    R2_TC_mod = C_SMOS_mod * C_mod_ASC / C_SMOS_ASC / C_mod_mod
    R2_TC_ASC = C_mod_ASC * C_SMOS_ASC / C_SMOS_mod / C_ASC_ASC

    for arr in (R2_TC_SMOS, R2_TC_mod, R2_TC_ASC):
        arr[arr < 0.001] = np.nan
        arr[arr > 1.0] = 1.0

    sigma2_SMOS = C_SMOS_SMOS - C_SMOS_ASC * C_SMOS_mod / C_mod_ASC
    sigma2_mod = C_mod_mod - C_mod_ASC * C_SMOS_mod / C_SMOS_ASC
    sigma2_ASC = C_ASC_ASC - C_SMOS_ASC * C_mod_ASC / C_SMOS_mod

    lons = lon_L4.reshape(-1, order='F')[:, None]
    lats = lat_L4.reshape(-1, order='F')[:, None]

    fout = cfg.ivs_output_root / f'ASCL4_SMOSIC_{run_name}_TC_stats_{tag}.mat'
    if not cfg.do_year_tc:
        fout = cfg.ivs_output_root / f'ASCL4_SMOSIC_{run_name}_TC_stats_{tag}_{"summer" if cfg.do_summer_tc else "winter"}.mat'

    sio.savemat(fout, {
        'lons': lons,
        'lats': lats,
        'N_sm': N_sm.astype(np.int32),
        'Nmin': np.array([[nmin]], dtype=np.int32),
        'R2_TC_SMOS': R2_TC_SMOS,
        'R2_TC_ASC': R2_TC_ASC,
        'R2_TC_mod': R2_TC_mod,
        'sigma2_SMOS': sigma2_SMOS,
        'sigma2_mod': sigma2_mod,
        'sigma2_ASC': sigma2_ASC,
        'R_mod_SMOS': R_mod_SMOS,
        'R_mod_ASC': R_mod_ASC,
        'R_ASC_SMOS': R_ASC_SMOS,
    }, do_compression=True)

    print(f'Wrote: {fout}')
    return fout


In [ ]:
# ---------- Step 5: Rdiff export ----------

def run_step5_rdiff(prefix: str, d1_version: str, d2_version: str, cfg: WorkflowConfig):
    tag = matlab_time_tag(cfg.start_date, cfg.end_date)
    f1 = cfg.ivs_output_root / f'{prefix}_{d1_version}_IVD_IVS_stats_lag{cfg.nlag_days}day_{tag}.mat'
    f2 = cfg.ivs_output_root / f'{prefix}_{d2_version}_IVD_IVS_stats_lag{cfg.nlag_days}day_{tag}.mat'

    d1 = load_mat_fields(f1, ('R2_ivs_obs', 'R2_ivs_mod'))
    d2 = load_mat_fields(f2, ('R2_ivs_obs', 'R2_ivs_mod'))

    R_D1 = np.sqrt(np.asarray(d1['R2_ivs_mod'], dtype=np.float64).reshape(-1))
    R_D2 = np.sqrt(np.asarray(d2['R2_ivs_mod'], dtype=np.float64).reshape(-1))
    R_OBS = np.sqrt(np.asarray(d2['R2_ivs_obs'], dtype=np.float64).reshape(-1))

    R_D1[R_D1 < 0.1] = np.nan
    R_D2[R_D2 < 0.1] = np.nan
    R_OBS[R_OBS < 0.1] = np.nan

    bad_obs = ~np.isfinite(R_OBS)
    R_D1[bad_obs] = np.nan
    R_D2[bad_obs] = np.nan

    Rdiff_vector = (R_D2 - R_D1).reshape(-1, 1)
    lons = lon_L4.reshape(-1, order='F')[:, None]
    lats = lat_L4.reshape(-1, order='F')[:, None]

    out = cfg.ivs_output_root / f'Rdiff_{d2_version}_minus_{d1_version}_{prefix}.mat'
    sio.savemat(out, {'Rdiff_vector': Rdiff_vector, 'lons': lons, 'lats': lats}, do_compression=True)

    print(f'Wrote: {out}')
    print(f'Mean ΔR = {np.nanmean(Rdiff_vector):.6f}')
    return out


In [ ]:
# ---------- Runner ----------
# Flip switches as needed.

RUN_STEP2_ASCL4 = False
RUN_STEP2_SMOSIC = False
RUN_STEP3 = False
RUN_STEP4 = False
RUN_TC = False
RUN_STEP5 = False

D1_VERSION = 'OLv8_M36_cd'
D2_VERSION = 'DAv8_M36_cd'

if RUN_STEP2_ASCL4:
    for run_name, run_root in cfg.run_roots.items():
        run_step2_for_sensor('ASCL4', run_name, run_root, cfg)

if RUN_STEP2_SMOSIC:
    for run_name, run_root in cfg.run_roots.items():
        run_step2_for_sensor('SMOSIC', run_name, run_root, cfg)

if RUN_STEP3:
    for run_name in cfg.run_roots:
        run_step3_climatology('ASCL4', run_name, cfg)
        run_step3_climatology('SMOSIC', run_name, cfg)

if RUN_STEP4:
    for run_name in cfg.run_roots:
        run_step4_ivd_ivs('ASCL4', run_name, cfg)
        run_step4_ivd_ivs('SMOSIC', run_name, cfg)

if RUN_TC:
    for run_name in cfg.run_roots:
        run_tc_ascat_smosic(run_name, cfg)

if RUN_STEP5:
    run_step5_rdiff('ASCL4', D1_VERSION, D2_VERSION, cfg)
    run_step5_rdiff('SMOSIC', D1_VERSION, D2_VERSION, cfg)

print('Done')
